# Common Libraries

In [1]:
import os, shutil, sys
if shutil.which('nvidia-smi') is not None:
    os.environ['MUJOCO_GL'] = 'egl'
import mujoco
import numpy as np
import ipywidgets as widgets
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider
from IPython.display import display, clear_output
from ipywidgets import FloatSlider, HBox, VBox, interactive_output, Label

# Params

In [2]:
# model_dir_path = "/home/seojin/myosuite/myosuite/simhive/myo_sim/arm"
# model_path = os.path.join(model_dir_path, "myoarm.xml")

model_dir_path = "/mnt/sdb2/DeepProprioception/Projects/DP01_mri/Myosuite/Model"
model_path = os.path.join(model_dir_path, "scaled_model_cvt3.xml")

# Functions

In [10]:
def render_at_pos(x, y, z, cam_dist, cam_azim, cam_elev):
    data.mocap_pos[mocap_id] = [x, y, z]
    mujoco.mj_forward(model, data)

    camera.distance = cam_dist
    camera.azimuth = cam_azim
    camera.elevation = cam_elev
    
    renderer.update_scene(data, camera=camera)
    pixels = renderer.render()

    clear_output(wait=True) 
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.imshow(pixels)
    ax.axis("off")
    ax.set_title(f"Mocap: [{x:.2f}, {y:.2f}, {z:.2f}] | Cam: Dist={cam_dist:.2f}, Azim={cam_azim:.2f}, Elev={cam_elev:.2f}")
    plt.show()

In [11]:
os.chdir(model_dir_path)

# Model specification

In [12]:
# Add mocap
xml_string = f"""
<mujoco model="MyoArm with Mocap">
    <include file="{model_path}"/>
    <worldbody>
        <body name="target" pos="0 0 0" quat="0 1 0 0" mocap="true">
            <geom type="box" size=".15 .15 .15" contype="0" conaffinity="0" rgba=".6 .3 .3 .2"/>
        </body>
    </worldbody>
</mujoco>
"""

with open(model_path, 'r') as f:
    xml_string = f.read()

mocap_xml = """
    <body name="target_mocap" mocap="true">
        <geom type="sphere" size="0.05" rgba="1 0 0 0.5" contype="0" conaffinity="0"/>
    </body>
"""
xml_string = xml_string.replace("</worldbody>", f"{mocap_xml}\n</worldbody>")

# Initialize model and data

In [13]:
model = mujoco.MjModel.from_xml_string(xml_string)
data = mujoco.MjData(model)
mocap_id = model.body_mocapid[mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "controller_mocap")]

if model.nkey:
    data.qpos = model.key_qpos[0]
mujoco.mj_forward(model, data)

# Initialize render

In [14]:
look_at_body_name = "thorax"

camera = mujoco.MjvCamera()
body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, look_at_body_name)
look_at_pos = data.xpos[body_id]
camera.lookat = data.mocap_pos[mocap_id]

renderer = mujoco.Renderer(model)

# UX

In [16]:
# UX
s_x = FloatSlider(min=-1.0, max=1.0, step=0.05, value=0.0, description='Mocap X', continuous_update=False)
s_y = FloatSlider(min=-1.0, max=1.0, step=0.05, value=0.0, description='Mocap Y', continuous_update=False)
s_z = FloatSlider(min=-1.0, max=1.0, step=0.05, value=0.0, description='Mocap Z', continuous_update=False)

c_dist = FloatSlider(min=1.0, max=10.0, step=0.1, value=2.0, description='Cam Dist', continuous_update=False)
c_azim = FloatSlider(min=-180, max=180, step=1, value=90, description='Cam Azim', continuous_update=False)
c_elev = FloatSlider(min=-90, max=90, step=1, value=-20, description='Cam Elev', continuous_update=False)

# Link UX to callback
out = interactive_output(render_at_pos, {
    "x": s_x, "y": s_y, "z": s_z,
    "cam_dist": c_dist, "cam_azim": c_azim, "cam_elev": c_elev
})

# Layout
mocap_ui = VBox([Label(value="[ Mocap Control ]"), s_x, s_y, s_z])
camera_ui = VBox([Label(value="[ Camera Control ]"), c_dist, c_azim, c_elev])
ui = VBox([
    HBox([mocap_ui, camera_ui]),
    out,
])

display(ui)